### SCPT to VS

In [1]:
# Import packages for use:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Times New Roman'
pd.set_option('display.max_columns', None)

"C:/Users/5_kp/OneDrive - UCLA IT Services/PhD in Civil Engineering UCLA/04 First Year 24-25/01 VSPDB/Source Files/VSPDB 1.0 Tables/"

"/Users/5_kp/Library/CloudStorage/OneDrive-UCLAITServices/PhD in Civil Engineering UCLA/04 First Year 24-25/01 VSPDB/Source Files/VSPDB 1.0 Tables/"

In [2]:
VSPDB_TABLE_DIR = r"/Users/5_kp/Library/CloudStorage/OneDrive-UCLAITServices/PhD in Civil Engineering UCLA/04 First Year 24-25/01 VSPDB/Source Files/VSPDB 1.0 Tables/"

SITE     = pd.read_csv(VSPDB_TABLE_DIR + 'site.csv').rename(columns={'name':'site_name','latitude':'site_latitude','longitude':'site_longitude'})
CITATION = pd.read_csv(VSPDB_TABLE_DIR + 'citation.csv')
CPT_META = pd.read_csv(VSPDB_TABLE_DIR + 'conePenetrationTestMeta.csv')
CPT_DATA = pd.read_csv(VSPDB_TABLE_DIR + 'conePenetrationTestArray.csv')
TT_META = pd.read_csv(VSPDB_TABLE_DIR + 'travelTimeMeta.csv')
TT_DATA = pd.read_csv(VSPDB_TABLE_DIR + 'travelTimeArray.csv')
DGWT_DATA = pd.read_csv(VSPDB_TABLE_DIR + 'DGWT.csv')
VEL_META = pd.read_csv(VSPDB_TABLE_DIR + 'velocityProfileMeta.csv')
VEL_DATA = pd.read_csv(VSPDB_TABLE_DIR + 'velocityProfileArray.csv')

try:
    REVIEW_DF = pd.read_csv("Traveltime_review.csv")
except:
    # It is verified that the travelTimeMeta_ID and site_ID is one-to-one relation
    REVIEW_DF = TT_META.copy()[['travelTimeMeta_ID','site_ID']] 
    REVIEW_DF.insert(2, 'status', ['Pending']*len(REVIEW_DF), True)
    REVIEW_DF.insert(3, 'VsZ_Direct_interpretation', [np.nan]*len(REVIEW_DF), True)
    REVIEW_DF.insert(4, 'VsZ_Global', [np.nan]*len(REVIEW_DF), True)
    REVIEW_DF.insert(5, 'VsZp_Direct_interpretation', [np.nan]*len(REVIEW_DF), True)
    REVIEW_DF.insert(6, 'VsZp_Global', [np.nan]*len(REVIEW_DF), True)
    REVIEW_DF.insert(7, 'comment', [np.nan]*len(REVIEW_DF), True)
    REVIEW_DF.to_csv("Traveltime_review.csv", index=False, header=True)

try: 
    VS_CPT_DF = pd.read_csv("SCPT_Vs.csv")
except:
    VS_CPT_DF = pd.DataFrame(columns=['travelTimeMeta_ID', 'Vs_model', 'CPT_depth'])
    VS_CPT_DF.to_csv("SCPT_Vs.csv", index=False, header=True)

try:
    VS_SLOPEBREAK_DF = pd.read_csv("SLOPEBREAK_Vs.csv")
except:
    VS_SLOPEBREAK_DF = pd.DataFrame(columns=['travelTimeMeta_ID', 'Velocity', 'Top_depth', 'Bottom_depth'])
    VS_SLOPEBREAK_DF.to_csv("SLOPEBREAK_Vs.csv", index=False, header=True)

META = REVIEW_DF[['travelTimeMeta_ID','site_ID']]
META = META.merge(CPT_META, on=['site_ID'], how='left')
META = META.merge(CITATION, on=['citation_ID'], how='left')
META.to_csv("META.csv", index=False, header=True)

### Calculate Vs using Robertson 2012

Robertson has correlation of ln(Vs) = 1.93 + 0.5 ln(Qtn) + 0.25 ln (\sigmav'/pa) + 0.63 Ic

In [3]:
from SCPT_to_Vs_Tool_V2 import Preprocessing_CPT, Robertson
fz, Qtn_lay_resampled, Ic_lay_resampled, Qtn_lay, Ic_lay, Qtn_inv, Ic_inv, CPT_qt, CPT_depth, ztop, zbot = Preprocessing_CPT(travelTimeMeta_ID = 520)
Vs_Robertson = Robertson(Qtn_lay_resampled, fz, Ic_lay_resampled)

### Calculate Vs using National Model

In [4]:
from SCPT_to_Vs_Tool_V2 import get_CPT
fz, Ic, Qtncs, CPT_depth, Qtn_inv, Ic_inv = get_CPT(travelTimeMeta_ID = 520)
index = (Qtncs > 0) & (fz > 0)  & (Ic < 4.0)

b = [0.27370871, 3.58137023, 2.48307888, 4.09688239]
f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b[1] * (Ic[index] - b[2]))))
global_velocity = b[0] * np.log(Qtncs[index]) + f_Ic_constrain * np.log(fz[index]) + b[3]

### Calculate Vs Site Specific

In [5]:
def weighted_velocity (x , lay_id_assign_valid, Qtncs_CPT_valid, fz_CPT_valid, Ic_CPT_valid, Vsi):

    f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-x[1] * (Ic_CPT_valid - x[2]))))

    cpt_velocity = x[0] * np.log(Qtncs_CPT_valid) + f_Ic_constrain * np.log(fz_CPT_valid) + x[3]

    cpt_slowness = 1 / np.exp(cpt_velocity)
    dz = np.repeat(0.05, len(cpt_velocity))
    layer_slowness = np.bincount(lay_id_assign_valid, weights = cpt_slowness * dz)
    layer_thickness = np.bincount(lay_id_assign_valid, weights = dz)
    layer_velocity = layer_thickness / layer_slowness

    residual = np.log(Vsi) - np.log(layer_velocity)

    return residual

In [6]:
from SCPT_to_Vs_Tool_V2 import Preprocessing_TT, get_CPT
from scipy.optimize import least_squares

TT, TT_depth, halfdepth, DTT, slowness_TT, slowness_TT_depth, ztop_TT, zbot_TT = Preprocessing_TT(travelTimeMeta_ID=120)
Vsi = (zbot_TT - ztop_TT)/(DTT/1000)
Vsi = Vsi[(Vsi > 50) & (Vsi < 1000)]
ztop_TT = ztop_TT[(Vsi > 50) & (Vsi < 1000)]
zbot_TT = zbot_TT[(Vsi > 50) & (Vsi < 1000)]

layer_ids = np.arange(len(Vsi))

fz, Ic, Qtncs, CPT_depth, Qtn_inv, Ic_inv = get_CPT(travelTimeMeta_ID = 120)

lay_id_assign = np.full(len(CPT_depth), -1)

# Vectorized matching
for i in range(len(layer_ids)):

    mask = ((CPT_depth >= ztop_TT[i]) & (CPT_depth < zbot_TT[i]))

    lay_id_assign[mask] = layer_ids[i]

index = (lay_id_assign > -0.1) & (Qtncs > 0) & (fz > 0)  & (Ic < 4.0)

b0 = [0.27370871, 3.58137023, 2.48307888, 4.09688239]
result = least_squares(weighted_velocity, b0, args=(lay_id_assign[index], Qtncs[index], fz[index], Ic[index], Vsi))

b1, b2, b3, a = result.x
print ("Optimized parameters: b1 =", b1, ", b2 =", b2, ", b3 =", b3, ", a =", a)

f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b2 * (Ic - b3))))

cpt_velocity = b1 * np.log(Qtncs) + f_Ic_constrain * np.log(fz) + a

Optimized parameters: b1 = 0.40635036632472016 , b2 = 15.87871981934479 , b3 = 1.8955982216956733 , a = 3.7331748119664856


### GUI for all four methods

In [7]:
from SCPT_to_Vs_Tool_V2 import Master, interleave
import ipywidgets as widgets
from ipywidgets import HTML, Layout, HBox, VBox, Dropdown, FloatText, Button, Textarea
from IPython.display import display

%matplotlib widget

# Setup
style = {'description_width': 'initial'}
starting_index = REVIEW_DF[REVIEW_DF['status'] == 'Pending'].index[0]
travelTimeMeta_ID = REVIEW_DF.iloc[starting_index]['travelTimeMeta_ID']
VsZ = REVIEW_DF.iloc[starting_index]['VsZ_Direct_interpretation'] 
VsZ_model = REVIEW_DF.iloc[starting_index]['VsZ_Global']
VsZp = REVIEW_DF.iloc[starting_index]['VsZp_Direct_interpretation']

velocity_updated = []
top_depth_updated = []
bottom_depth_updated = []

TRAVELTIME = Dropdown(options=REVIEW_DF['travelTimeMeta_ID'].unique(), value=travelTimeMeta_ID, description='Travel Time ID', style=style)
VsZ_input = FloatText(value=VsZ, description='VsZ Direct Interpretation', style=style)
VsZ_model_input = FloatText(value=VsZ_model, description='VsZ National Model', style=style)
VsZp_input = FloatText(value=VsZp, description='VsZp Direct Interpretation', style=style)

button_layout = widgets.Layout(width='300px', height='40px')
ACCEPT = Button(description='Accept', layout = button_layout)
REVISE_FULL_HAND = Button(description='Full Manual Revision', layout = button_layout)
REJECT = Button(description='Reject', layout = button_layout)
COMMENT = Textarea(description='Review Comments:', value=None, style=style, layout=Layout(width='30%'))

# Dynamic checkbox container (empty to start)
checkbox_box = widgets.VBox(layout=Layout(flex_flow='row wrap', width='100%'))

box_layout_2 = widgets.Layout(width='100%', border='solid 2px', padding='20px')
SEARCH = VBox([
    HTML('<center><font size="+1.5"><b>SCPT Travel Time Interpretation Automated Tool</b>'),
    HBox([TRAVELTIME
          ], layout=Layout(width='100%')),
    HBox([VsZ_input, VsZ_model_input], layout=Layout(width='100%')),
    HBox([VsZp_input], layout=Layout(width='100%')),
    HBox([ACCEPT, 
          REVISE_FULL_HAND, 
        #   REVISE_SEMI_AUTO, 
          REJECT, COMMENT], layout=Layout(width='100%')),
    checkbox_box  # add the checkboxes below the buttons
], layout=box_layout_2)

out = widgets.Output()

# --- Reactive SCPT update ---
def Update_SCPT(_):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    out.clear_output()
    with out:
        ID = TRAVELTIME.value

        # Run main computation and get breaks
        Qtn_inv, CPT_I, Qtn_lay, ztop_I, zbot_I, Ic_I, Ic_lay, vs_lay, velocity, TTfromslowness, halfdepth, DTT, TT, TT_depth, results, top_depth, bottom_depth = Master(travelTimeMeta_ID=ID, slope_break_method=0)
        breaks = sorted(results.get('candidate_breakpoints', []))
        used_breaks = [str(round(b, 3)) for b in results.get('breaks', [])]
        

        print(f"Selected ID: {ID}")

        traveltimeMeta_ID = ID

        from SCPT_to_Vs_Tool_V2 import Preprocessing_CPT, Robertson
        fz_roberton, Qtn_robertson, Ic_robertson, Qtn_lay, Ic_lay, Qtn_inv, Ic_inv, CPT_qt, CPT_depth, ztop, zbot = Preprocessing_CPT(travelTimeMeta_ID = ID)
        Vs_Robertson = Robertson(Qtn_inv, fz_roberton, Ic_inv)


        from SCPT_to_Vs_Tool_V2 import get_CPT
        fz, Ic, Qtncs, CPT_depth, Qtn_inv, Ic_inv = get_CPT(travelTimeMeta_ID = ID)
        index1 = (Qtncs > 0) & (fz > 0)  & (Ic < 4.0)

        b = [0.27370871, 3.58137023, 2.48307888, 4.09688239]
        f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b[1] * (Ic[index1] - b[2]))))
        global_velocity = b[0] * np.log(Qtncs[index1]) + f_Ic_constrain * np.log(fz[index1]) + b[3]

        from SCPT_to_Vs_Tool_V2 import Preprocessing_TT, get_CPT
        from scipy.optimize import least_squares

        TT, TT_depth, halfdepth, DTT, slowness_TT, slowness_TT_depth, ztop_TT, zbot_TT = Preprocessing_TT(travelTimeMeta_ID=ID)
        Vsi = (zbot_TT - ztop_TT)/(DTT/1000)
        Vsi = Vsi[(Vsi > 50) & (Vsi < 1000)]
        ztop_TT = ztop_TT[(Vsi > 50) & (Vsi < 1000)]
        zbot_TT = zbot_TT[(Vsi > 50) & (Vsi < 1000)]

        layer_ids = np.arange(len(Vsi))

        fz, Ic, Qtncs, CPT_depth, Qtn_inv, Ic_inv = get_CPT(travelTimeMeta_ID = ID)

        lay_id_assign = np.full(len(CPT_depth), -1)

        # Vectorized matching
        for i in range(len(layer_ids)):

            mask = ((CPT_depth >= ztop_TT[i]) & (CPT_depth < zbot_TT[i]))

            lay_id_assign[mask] = layer_ids[i]

        index2 = (lay_id_assign > -0.1) & (Qtncs > 0) & (fz > 0)  & (Ic < 4.0)

        b0 = [0.27370871, 3.58137023, 2.48307888, 4.09688239]
        result = least_squares(weighted_velocity, b0, args=(lay_id_assign[index2], Qtncs[index2], fz[index2], Ic[index2], Vsi))

        b1, b2, b3, a = result.x
        print ("Site specific parameters: b1 =", b1, ", b2 =", b2, ", b3 =", b3, ", a =", a)

        f_Ic_constrain_1 = 0.1 + 0.4 * (1 / (1 + np.exp(-b2 * (Ic[index2] - b3))))
        cpt_velocity_1 = b1 * np.log(Qtncs[index2]) + f_Ic_constrain_1 * np.log(fz[index2]) + a

        f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b2 * (Ic - b3))))
        cpt_velocity = b1 * np.log(Qtncs) + f_Ic_constrain * np.log(fz) + a

        cpt_slowness = 1 / np.exp(cpt_velocity_1)
        dz = np.repeat(0.05, len(cpt_velocity_1))
        layer_slowness = np.bincount(lay_id_assign[index2], weights = cpt_slowness * dz)
        layer_thickness = np.bincount(lay_id_assign[index2], weights = dz)
        layer_velocity = layer_thickness / layer_slowness

        Qtn_plot = Qtncs[index2]
        CPT_depth_plot = CPT_depth[index2]
        fz_CPT_plot = fz[index2]
        Ic_CPT_plot = Ic[index2]
        ztop_plot = ztop_TT
        zbot_plot = zbot_TT

        TT_data = TT_DATA[TT_DATA['travelTimeMeta_ID'] == ID]
        TT_depth_plot = TT_data['depth']
        TT_time_plot = TT_data['traveltime']


        # using subplot to plot qt, fz, Ic, Vs with depth for the selected traveltimeMeta_ID
        fig, axs = plt.subplots(1, 6, figsize=(12, 5), dpi=100)

        axs[0].plot(Qtncs, CPT_depth, label='Qtncs', color = 'black')
        axs[0].plot(Qtn_inv, CPT_depth, label='Qtn')
        # axs[0].plot(interleave(qt_plot, qt_plot), interleave(ztop_plot, zbot_plot), label='qt_TT Depth Averaged', color='black')
        axs[0].set_xlim(0, 800)
        axs[0].set_xlabel('Qtn, Qtn,cs')
        axs[0].set_ylabel('Depth (m)')
        axs[0].set_title('Qtn, Qtn,cs vs Depth')
        axs[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        axs[1].plot(fz, CPT_depth, label='$\\sigma_v^\\prime$_CPT', color='black')
        axs[1].set_xlabel('$\\sigma_v^\\prime$ (atm)')
        axs[1].set_title('$\\sigma_v^\\prime$ vs Depth')
        axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        axs[2].plot(Ic, CPT_depth, label='Ic_layered', color = 'black')
        axs[2].plot(Ic_inv, CPT_depth, label='Ic')
        axs[2].set_xlim(1, 4)
        axs[2].set_xlabel('Ic')
        axs[2].set_title('Ic vs Depth')
        axs[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        # axs[3].plot(interleave(np.exp(Vs_pred_plot), np.exp(Vs_pred_plot)), interleave(ztop_plot, zbot_plot), label='Vsi_Model1', color='green')
        # axs[3].plot(np.exp(Vs_plot), CPT_depth_plot, label='Vs_model', color='orange')
        axs[3].plot(interleave(Vsi, Vsi), interleave(ztop_plot, zbot_plot), label='Vsi_Measured', color='black')
        axs[3].plot(interleave(layer_velocity, layer_velocity), interleave(ztop_plot, zbot_plot), label='Vsi_model', color='green')
        axs[3].set_xlim(0, 600)
        axs[3].set_xlabel('Velocity (m/s)')
        axs[3].set_title('Interval Velocity vs Depth')
        axs[3].grid(True, which="both", ls="--")
        axs[3].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        axs[4].scatter(TT, TT_depth_plot, label='Travel Time Data', color='black', s=10)

        colors = plt.get_cmap('tab10').colors

        # Plot each segment with a different color
        for i in range(len(results['breaks']) - 1):
            # Define segment depth range
            z_start = results['breaks'][i]
            z_end = results['breaks'][i + 1]

            # Get depths in this segment (new_depths ensures breaks are included)
            segment_mask = (results['new_depths'] >= z_start) & (results['new_depths'] <= z_end)
            segment_depths = results['new_depths'][segment_mask]
            segment_tt = results['fitted_values_for_plot'][segment_mask]

            # Plot segment
            axs[4].plot(segment_tt, segment_depths, color=colors[i], 
                        # label=f'Segment {i+1}'
                        )

        axs[4].set_xlabel('Travel Time (ms)')
        axs[4].set_xlim(0, np.max(TT_time_plot) * 1.1)
        if np.max(TT_time_plot) > 150:
            axs[4].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 50))
        elif np.max(TT_time_plot) > 90:
            axs[4].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 30))
        elif np.max(TT_time_plot) > 50:
            axs[4].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 20))
        else:
            axs[4].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 10))
        axs[4].set_title('Travel Time vs Depth')
        axs[4].grid(True, which="both", ls="--")
        axs[4].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)


        axs[5].plot(Vs_Robertson, CPT_depth, color='red', label = 'Robertson (2012)', alpha=0.5)
        axs[5].plot(np.exp(global_velocity), CPT_depth[index1], label='National Model', color='orange')
        axs[5].plot(np.exp(cpt_velocity), CPT_depth, label='Site Specific Model', color='green')
        axs[5].plot(interleave(velocity, velocity), interleave(top_depth, bottom_depth), label='Direct Interpreted Velocity', color='blue')

        axs[5].set_xlim(0, 600)
        axs[5].set_xticks(np.arange(0, 601, 100))

        axs[5].set_xlabel('Velocity (m/s)')
        axs[5].set_title('Velocity vs Depth')
        axs[5].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[5].grid(True, which="both", ls="--")


        # Share common y-axis limits across all subplots
        y_min = np.nanmin([np.nanmin(CPT_depth_plot), np.nanmin(ztop_plot), np.nanmin(TT_depth_plot)])
        y_max = np.nanmax([np.nanmax(CPT_depth_plot), np.nanmax(zbot_plot), np.nanmax(TT_depth_plot)])
        for ax in axs:
            ax.set_ylim(y_max + 1, 0)  # inverted depth axis, same range for all

        plt.subplots_adjust(bottom=0.28, wspace=0.35)
        plt.tight_layout()
        plt.show()

        # Calculate VsZ for direct interpretation
        top_depth_dir = np.asarray(top_depth)
        top_depth_dir[0] = 0.0
        VsZ = np.max(np.asarray(bottom_depth)) / np.sum((np.asarray(bottom_depth) - top_depth_dir) / np.asarray(velocity))
        # print(f"Directly Interpreted VsZ: {VsZ:.2f} m/s to depth {np.max(np.asarray(bottom_depth)):.2f} m")

        # Calculate VsZ for global model
        VsZ_ss_top_depth = np.insert(np.asarray(CPT_depth[index1][:-1]), 0, 0.0)
        VsZ_bad = np.max(CPT_depth[index1]) / np.sum((np.asarray(CPT_depth[index1]) - VsZ_ss_top_depth) / np.asarray(np.exp(global_velocity)))
        # print(f"Global Model VsZ: {VsZ_bad:.2f} m/s to depth {np.max(CPT_depth[index1]):.2f} m")

        # # Calculate VsZ for direct interpretation between the first TT_depth_plot to the maximum depth of the interpreted velocity model
        top_depth_lim = np.asarray(top_depth)
        top_depth_lim [0] = np.min(TT_depth_plot)
        VsZp = (np.max(np.asarray(TT_depth_plot))-np.min(np.asarray(TT_depth_plot))) / np.sum((np.asarray(bottom_depth) - top_depth_lim) / np.asarray(velocity))
        # print(f"Directly Interpreted VsZ from the top depth of the interpreted velocity model: {VsZp:.2f} m/s to depth {np.max(np.asarray(TT_depth_plot)):.2f} m")


        # Update the checkboxes dynamically
        new_checkboxes = [
            widgets.Checkbox(description=str(round(b, 3)), indent=False, value=(str(round(b, 3)) in used_breaks))
            for b in sorted(breaks)
        ]
        checkbox_box.children = new_checkboxes

        VsZ_input.value = VsZ
        VsZ_model_input.value = VsZ_bad
        VsZp_input.value = VsZp

        velocity_updated = velocity
        top_depth_updated = top_depth
        bottom_depth_updated = bottom_depth

def Full_Manual(_):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    out.clear_output()
    with out:
        # Get selected breakpoints
        selected_breakpoints = [float(checkbox.description) for checkbox in checkbox_box.children if checkbox.value]
        if not selected_breakpoints:
            print("No breakpoints selected.")
            return
        # Run full manual regression
        Qtn_inv, CPT_I, Qtn_lay, ztop_I, zbot_I, Ic_I, Ic_lay, vs_lay, velocity, TTfromslowness, halfdepth, DTT, TT, TT_depth, results, top_depth, bottom_depth = Master(travelTimeMeta_ID=TRAVELTIME.value, slope_break_method=1, breakpoints=selected_breakpoints)
        print(f"Selected ID: {TRAVELTIME.value}")
        traveltimeMeta_ID = TRAVELTIME.value

        ID = traveltimeMeta_ID

        from SCPT_to_Vs_Tool_V2 import Preprocessing_CPT, Robertson
        fz_roberton, Qtn_robertson, Ic_robertson, Qtn_lay, Ic_lay, Qtn_inv, Ic_inv, CPT_qt, CPT_depth, ztop, zbot = Preprocessing_CPT(travelTimeMeta_ID = ID)
        Vs_Robertson = Robertson(Qtn_inv, fz_roberton, Ic_inv)

        from SCPT_to_Vs_Tool_V2 import get_CPT
        fz, Ic, Qtncs, CPT_depth, Qtn_inv, Ic_inv = get_CPT(travelTimeMeta_ID = ID)
        index1 = (Qtncs > 0) & (fz > 0)  & (Ic < 4.0)

        b = [0.27370871, 3.58137023, 2.48307888, 4.09688239]
        f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b[1] * (Ic[index1] - b[2]))))
        global_velocity = b[0] * np.log(Qtncs[index1]) + f_Ic_constrain * np.log(fz[index1]) + b[3]

        from SCPT_to_Vs_Tool_V2 import Preprocessing_TT, get_CPT
        from scipy.optimize import least_squares

        TT, TT_depth, halfdepth, DTT, slowness_TT, slowness_TT_depth, ztop_TT, zbot_TT = Preprocessing_TT(travelTimeMeta_ID=ID)
        Vsi = (zbot_TT - ztop_TT)/(DTT/1000)
        Vsi = Vsi[(Vsi > 50) & (Vsi < 1000)]
        ztop_TT = ztop_TT[(Vsi > 50) & (Vsi < 1000)]
        zbot_TT = zbot_TT[(Vsi > 50) & (Vsi < 1000)]

        layer_ids = np.arange(len(Vsi))

        fz, Ic, Qtncs, CPT_depth, Qtn_inv, Ic_inv = get_CPT(travelTimeMeta_ID = ID)

        lay_id_assign = np.full(len(CPT_depth), -1)

        # Vectorized matching
        for i in range(len(layer_ids)):

            mask = ((CPT_depth >= ztop_TT[i]) & (CPT_depth < zbot_TT[i]))

            lay_id_assign[mask] = layer_ids[i]

        index2 = (lay_id_assign > -0.1) & (Qtncs > 0) & (fz > 0)  & (Ic < 4.0)

        b0 = [0.27370871, 3.58137023, 2.48307888, 4.09688239]
        result = least_squares(weighted_velocity, b0, args=(lay_id_assign[index2], Qtncs[index2], fz[index2], Ic[index2], Vsi))

        b1, b2, b3, a = result.x
        print ("Site specific parameters: b1 =", b1, ", b2 =", b2, ", b3 =", b3, ", a =", a)

        f_Ic_constrain_1 = 0.1 + 0.4 * (1 / (1 + np.exp(-b2 * (Ic[index2] - b3))))
        cpt_velocity_1 = b1 * np.log(Qtncs[index2]) + f_Ic_constrain_1 * np.log(fz[index2]) + a

        f_Ic_constrain = 0.1 + 0.4 * (1 / (1 + np.exp(-b2 * (Ic - b3))))
        cpt_velocity = b1 * np.log(Qtncs) + f_Ic_constrain * np.log(fz) + a

        cpt_slowness = 1 / np.exp(cpt_velocity_1)
        dz = np.repeat(0.05, len(cpt_velocity_1))
        layer_slowness = np.bincount(lay_id_assign[index2], weights = cpt_slowness * dz)
        layer_thickness = np.bincount(lay_id_assign[index2], weights = dz)
        layer_velocity = layer_thickness / layer_slowness

        Qtn_plot = Qtncs[index2]
        CPT_depth_plot = CPT_depth[index2]
        fz_CPT_plot = fz[index2]
        Ic_CPT_plot = Ic[index2]
        ztop_plot = ztop_TT
        zbot_plot = zbot_TT

        TT_data = TT_DATA[TT_DATA['travelTimeMeta_ID'] == ID]
        TT_depth_plot = TT_data['depth']
        TT_time_plot = TT_data['traveltime']


        # using subplot to plot qt, fz, Ic, Vs with depth for the selected traveltimeMeta_ID
        fig, axs = plt.subplots(1, 6, figsize=(12, 5), dpi=100)

        axs[0].plot(Qtncs, CPT_depth, label='Qtncs', color = 'black')
        axs[0].plot(Qtn_inv, CPT_depth, label='Qtn_inv')
        axs[0].set_xlim(0, 800)
        axs[0].set_xlabel('Qtn, Qtn,cs')
        axs[0].set_ylabel('Depth (m)')
        axs[0].set_title('Qtn, Qtn,cs vs Depth')
        axs[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        axs[1].plot(fz, CPT_depth, label='$\\sigma_v^\\prime$_CPT', color='black')
        axs[1].set_xlabel('$\\sigma_v^\\prime$ (atm)')
        axs[1].set_title('$\\sigma_v^\\prime$ vs Depth')
        axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        axs[2].plot(Ic, CPT_depth, label='Ic_layered', color = 'black')
        axs[2].plot(Ic_inv, CPT_depth, label='Ic_inv')
        axs[2].set_xlim(1, 4)
        axs[2].set_xlabel('Ic')
        axs[2].set_title('Ic vs Depth')
        axs[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        # axs[3].plot(interleave(np.exp(Vs_pred_plot), np.exp(Vs_pred_plot)), interleave(ztop_plot, zbot_plot), label='Vsi_Model1', color='green')
        # axs[3].plot(np.exp(Vs_plot), CPT_depth_plot, label='Vs_model', color='orange')
        axs[3].plot(interleave(Vsi, Vsi), interleave(ztop_plot, zbot_plot), label='Vsi_Measured', color='black')
        axs[3].plot(interleave(layer_velocity, layer_velocity), interleave(ztop_plot, zbot_plot), label='Vsi_Model3', color='green')
        axs[3].set_xlim(0, 600)
        axs[3].set_xlabel('Velocity (m/s)')
        axs[3].set_title('Interval Velocity vs Depth')
        axs[3].grid(True, which="both", ls="--")
        axs[3].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        axs[4].scatter(TT, TT_depth_plot, label='Travel Time Data', color='black', s=10)

        colors = plt.get_cmap('tab10').colors

        # Plot each segment with a different color
        for i in range(len(results['breaks']) - 1):
            # Define segment depth range
            z_start = results['breaks'][i]
            z_end = results['breaks'][i + 1]

            # Get depths in this segment (new_depths ensures breaks are included)
            segment_mask = (results['new_depths'] >= z_start) & (results['new_depths'] <= z_end)
            segment_depths = results['new_depths'][segment_mask]
            segment_tt = results['fitted_values_for_plot'][segment_mask]

            # Plot segment
            axs[4].plot(segment_tt, segment_depths, color=colors[i], 
                        # label=f'Segment {i+1}'
                        )

        axs[4].set_xlabel('Travel Time (ms)')
        axs[4].set_xlim(0, np.max(TT_time_plot) * 1.1)
        if np.max(TT_time_plot) > 150:
            axs[4].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 50))
        elif np.max(TT_time_plot) > 90:
            axs[4].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 30))
        elif np.max(TT_time_plot) > 50:
            axs[4].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 20))
        else:
            axs[4].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 10))
        axs[4].set_title('Travel Time vs Depth')
        axs[4].grid(True, which="both", ls="--")
        axs[4].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        # axs[5].plot(np.exp(Vs_plot), CPT_depth_plot, label='Coefficients from regression', color='green')
        # axs[5].plot(np.exp(bad_Vs_plot), bad_depth_plot, color='red', label = 'Model')
        # axs[5].plot(Vs_Robertson, CPT_depth, color='red', label = 'Robertson (2012)', alpha=0.5)
        # axs[5].plot(np.exp(global_velocity), CPT_depth[index1], label='National Model', color='orange')
        axs[5].plot(np.exp(cpt_velocity), CPT_depth, label='Site Specific Model', color='green')
        # axs[5].plot(interleave(velocity, velocity), interleave(top_depth, bottom_depth), label='Direct Interpreted Velocity', color='blue')

        axs[5].set_xlim(0, 600)
        axs[5].set_xticks(np.arange(0, 601, 100))

        axs[5].set_xlabel('Velocity (m/s)')
        axs[5].set_title('Velocity vs Depth')
        axs[5].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[5].grid(True, which="both", ls="--")


        # Share common y-axis limits across all subplots
        y_min = np.nanmin([np.nanmin(CPT_depth_plot), np.nanmin(ztop_plot), np.nanmin(TT_depth_plot)])
        y_max = np.nanmax([np.nanmax(CPT_depth_plot), np.nanmax(zbot_plot), np.nanmax(TT_depth_plot)])
        for ax in axs:
            ax.set_ylim(y_max + 1, 0)  # inverted depth axis, same range for all

        plt.subplots_adjust(bottom=0.28, wspace=0.35)
        plt.tight_layout()
        plt.show()

        # Calculate VsZ for direct interpretation
        top_depth_dir = np.asarray(top_depth)
        top_depth_dir[0] = 0.0
        VsZ = np.max(np.asarray(bottom_depth)) / np.sum((np.asarray(bottom_depth) - top_depth_dir) / np.asarray(velocity))
        print(f"Directly Interpreted VsZ: {VsZ:.2f} m/s to depth {np.max(np.asarray(bottom_depth)):.2f} m")

        # Calculate VsZ for global model
        VsZ_ss_top_depth = np.insert(np.asarray(CPT_depth[index1][:-1]), 0, 0.0)
        VsZ_bad = np.max(CPT_depth[index1]) / np.sum((np.asarray(CPT_depth[index1]) - VsZ_ss_top_depth) / np.asarray(np.exp(global_velocity)))
        print(f"Global Model VsZ: {VsZ_bad:.2f} m/s to depth {np.max(CPT_depth[index1]):.2f} m")

        # # Calculate VsZ for direct interpretation between the first TT_depth_plot to the maximum depth of the interpreted velocity model
        top_depth_lim = np.asarray(top_depth)
        top_depth_lim [0] = np.min(TT_depth_plot)
        VsZp = (np.max(np.asarray(TT_depth_plot))-np.min(np.asarray(TT_depth_plot))) / np.sum((np.asarray(bottom_depth) - top_depth_lim) / np.asarray(velocity))
        print(f"Directly Interpreted VsZ from the top depth of the interpreted velocity model: {VsZp:.2f} m/s to depth {np.max(np.asarray(TT_depth_plot)):.2f} m")

        VsZ_input.value = VsZ
        VsZ_model_input.value = VsZ_bad
        VsZp_input.value = VsZp

        velocity_updated = velocity
        top_depth_updated = top_depth
        bottom_depth_updated = bottom_depth



# Accept Function
def Accept (_):
    
    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    # Save VsZ and velocity profile for the accepted model
    travelTimeMeta_ID = TRAVELTIME.value
    VsZ = VsZ_input.value
    VsZ_model = VsZ_model_input.value
    VsZp = VsZp_input.value
    comment = COMMENT.value

    # Update the DataFrame
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['status']] = ['accepted']
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['VsZ_Direct_interpretation']] = VsZ
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['VsZ_Global']] = VsZ_model
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['VsZp_Direct_interpretation']] = VsZp
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['comment']] = comment

    # Save to CSV
    REVIEW_DF.to_csv('Traveltime_review.csv', index=False)

    try:
        VS_SLOPEBREAK_DF = pd.read_csv('SLOPEBREAK_Vs.csv')
    except FileNotFoundError:
        VS_SLOPEBREAK_DF = pd.DataFrame(columns=['travelTimeMeta_ID', 'Velocity', 'Top_depth', 'Bottom_depth'])

    velocity_save = velocity_updated
    top_depth_save = top_depth_updated
    bottom_depth_save = bottom_depth_updated


    # Save the Vs profile to a CSV file
    TT_ID_array2 = np.full(len(velocity_save), travelTimeMeta_ID)
    new_data2 = pd.DataFrame({'travelTimeMeta_ID': TT_ID_array2,
    'Velocity': velocity_save,
    'Top_depth': top_depth_save,
    'Bottom_depth': bottom_depth_save})
    VS_SLOPEBREAK_DF = pd.concat([VS_SLOPEBREAK_DF, new_data2], ignore_index=True)
    # Save to CSV
    VS_SLOPEBREAK_DF.to_csv('SLOPEBREAK_Vs.csv', index=False)

    # Move to the next ID
    try:
        next_index = REVIEW_DF[REVIEW_DF['status'] == 'Pending']['travelTimeMeta_ID'].values[0]
        TRAVELTIME.value = next_index
    except:
        print("No more IDs to process.")

def Reject (_):
    # Update the DataFrame
    travelTimeMeta_ID = TRAVELTIME.value
    comment = COMMENT.value
    # Update the status and comment: invalid CPT or Traveltime data
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['status', 'comment']] = ['rejected', comment]
    # Save to CSV
    REVIEW_DF.to_csv('Traveltime_review.csv', index=False)


# Initial display
display(SEARCH)
display(out)

# Trigger first update manually
Update_SCPT(None)

# Attach trigger to dropdown
TRAVELTIME.observe(Update_SCPT, names=['value'])
ACCEPT.on_click(Accept)
REVISE_FULL_HAND.on_click(Full_Manual)
REJECT.on_click(Reject)

/Users/5_kp/Library/Python/3.9/lib/python/site-packages/jupyter_client/session.py:721: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


Output()

In [12]:
from SCPT_to_Vs_Tool_V2 import Master, interleave
import ipywidgets as widgets
from ipywidgets import HTML, Layout, HBox, VBox, Dropdown, FloatText, Button, Textarea
from IPython.display import display

%matplotlib widget

# Setup
style = {'description_width': 'initial'}
starting_index = REVIEW_DF[REVIEW_DF['status'] == 'Pending'].index[0]
travelTimeMeta_ID = REVIEW_DF.iloc[starting_index]['travelTimeMeta_ID']
VsZ = REVIEW_DF.iloc[starting_index]['VsZ_Direct_interpretation'] 
VsZp = REVIEW_DF.iloc[starting_index]['VsZp_Direct_interpretation']

velocity_updated = []
top_depth_updated = []
bottom_depth_updated = []

TRAVELTIME = Dropdown(options=REVIEW_DF['travelTimeMeta_ID'].unique(), value=travelTimeMeta_ID, description='Travel Time ID', style=style)
VsZ_input = FloatText(value=VsZ, description='VsZ Direct Interpretation', style=style)
VsZp_input = FloatText(value=VsZp, description='VsZp Direct Interpretation', style=style)

button_layout = widgets.Layout(width='300px', height='40px')
ACCEPT = Button(description='Accept', layout = button_layout)
REVISE_FULL_HAND = Button(description='Full Manual Revision', layout = button_layout)
REJECT = Button(description='Reject', layout = button_layout)
COMMENT = Textarea(description='Review Comments:', value=None, style=style, layout=Layout(width='30%'))

# Dynamic checkbox container (empty to start)
checkbox_box = widgets.VBox(layout=Layout(flex_flow='row wrap', width='100%'))

box_layout_2 = widgets.Layout(width='100%', border='solid 2px', padding='20px')
SEARCH = VBox([
    HTML('<center><font size="+1.5"><b>SCPT Travel Time Interpretation Automated Tool</b>'),
    HBox([TRAVELTIME
          ], layout=Layout(width='100%')),
    HBox([VsZ_input], layout=Layout(width='100%')),
    HBox([VsZp_input], layout=Layout(width='100%')),
    HBox([ACCEPT, 
          REVISE_FULL_HAND, 
        #   REVISE_SEMI_AUTO, 
          REJECT, COMMENT], layout=Layout(width='100%')),
    checkbox_box  # add the checkboxes below the buttons
], layout=box_layout_2)

out = widgets.Output()

# --- Reactive SCPT update ---
def Update_SCPT(_):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    out.clear_output()
    with out:
        ID = TRAVELTIME.value

        # Run main computation and get breaks
        Qtn_inv, CPT_I, Qtn_lay, ztop_I, zbot_I, Ic_I, Ic_lay, vs_lay, velocity, TTfromslowness, halfdepth, DTT, TT, TT_depth, results, top_depth, bottom_depth = Master(travelTimeMeta_ID=ID, slope_break_method=0)
        breaks = sorted(results.get('candidate_breakpoints', []))
        used_breaks = [str(round(b, 3)) for b in results.get('breaks', [])]
        

        print(f"Selected ID: {ID}")

        traveltimeMeta_ID = ID

        TT_data = TT_DATA[TT_DATA['travelTimeMeta_ID'] == ID]
        TT_depth_plot = TT_data['depth']
        TT_time_plot = TT_data['traveltime']


        # using subplot to plot qt, fz, Ic, Vs with depth for the selected traveltimeMeta_ID
        fig, axs = plt.subplots(1, 2, figsize=(6, 5), dpi=100)

        axs[0].scatter(TT, TT_depth_plot, label='Travel Time Data', color='black', s=10)

        colors = plt.get_cmap('tab10').colors

        # Plot each segment with a different color
        for i in range(len(results['breaks']) - 1):
            # Define segment depth range
            z_start = results['breaks'][i]
            z_end = results['breaks'][i + 1]

            # Get depths in this segment (new_depths ensures breaks are included)
            segment_mask = (results['new_depths'] >= z_start) & (results['new_depths'] <= z_end)
            segment_depths = results['new_depths'][segment_mask]
            segment_tt = results['fitted_values_for_plot'][segment_mask]

            # Plot segment
            axs[0].plot(segment_tt, segment_depths, color=colors[i], 
                        # label=f'Segment {i+1}'
                        )

        axs[0].set_xlabel('Travel Time (ms)')
        axs[0].set_xlim(0, np.max(TT_time_plot) * 1.1)
        if np.max(TT_time_plot) > 150:
            axs[0].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 50))
        elif np.max(TT_time_plot) > 90:
            axs[0].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 30))
        elif np.max(TT_time_plot) > 50:
            axs[0].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 20))
        else:
            axs[0].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 10))
        axs[0].set_title('Travel Time vs Depth')
        axs[0].grid(True, which="both", ls="--")
        axs[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)


        axs[1].plot(interleave(velocity, velocity), interleave(top_depth, bottom_depth), label='Direct Interpreted Velocity', color='blue')

        axs[1].set_xlim(0, 600)
        axs[1].set_xticks(np.arange(0, 601, 100))

        axs[1].set_xlabel('Velocity (m/s)')
        axs[1].set_title('Velocity vs Depth')
        axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[1].grid(True, which="both", ls="--")


        # Share common y-axis limits across all subplots
        y_max = np.nanmax(bottom_depth)  # Use the maximum depth from the interpreted model for consistent y-axis limits
        for ax in axs:
            ax.set_ylim(y_max + 1, 0)  # inverted depth axis, same range for all

        plt.subplots_adjust(bottom=0.28, wspace=0.35)
        plt.tight_layout()
        plt.show()

        # Calculate VsZ for direct interpretation
        top_depth_dir = np.asarray(top_depth)
        top_depth_dir[0] = 0.0
        VsZ = np.max(np.asarray(bottom_depth)) / np.sum((np.asarray(bottom_depth) - top_depth_dir) / np.asarray(velocity))
        # print(f"Directly Interpreted VsZ: {VsZ:.2f} m/s to depth {np.max(np.asarray(bottom_depth)):.2f} m")


        # # Calculate VsZ for direct interpretation between the first TT_depth_plot to the maximum depth of the interpreted velocity model
        top_depth_lim = np.asarray(top_depth)
        top_depth_lim [0] = np.min(TT_depth_plot)
        VsZp = (np.max(np.asarray(TT_depth_plot))-np.min(np.asarray(TT_depth_plot))) / np.sum((np.asarray(bottom_depth) - top_depth_lim) / np.asarray(velocity))
        # print(f"Directly Interpreted VsZ from the top depth of the interpreted velocity model: {VsZp:.2f} m/s to depth {np.max(np.asarray(TT_depth_plot)):.2f} m")


        # Update the checkboxes dynamically
        new_checkboxes = [
            widgets.Checkbox(description=str(round(b, 3)), indent=False, value=(str(round(b, 3)) in used_breaks))
            for b in sorted(breaks)
        ]
        checkbox_box.children = new_checkboxes

        VsZ_input.value = VsZ
        VsZp_input.value = VsZp

        velocity_updated = velocity
        top_depth_updated = top_depth
        bottom_depth_updated = bottom_depth

def Full_Manual(_):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    out.clear_output()
    with out:
        # Get selected breakpoints
        selected_breakpoints = [float(checkbox.description) for checkbox in checkbox_box.children if checkbox.value]
        if not selected_breakpoints:
            print("No breakpoints selected.")
            return
        # Run full manual regression
        Qtn_inv, CPT_I, Qtn_lay, ztop_I, zbot_I, Ic_I, Ic_lay, vs_lay, velocity, TTfromslowness, halfdepth, DTT, TT, TT_depth, results, top_depth, bottom_depth = Master(travelTimeMeta_ID=TRAVELTIME.value, slope_break_method=1, breakpoints=selected_breakpoints)
        print(f"Selected ID: {TRAVELTIME.value}")
        traveltimeMeta_ID = TRAVELTIME.value

        ID = traveltimeMeta_ID

        TT_data = TT_DATA[TT_DATA['travelTimeMeta_ID'] == ID]
        TT_depth_plot = TT_data['depth']
        TT_time_plot = TT_data['traveltime']


        # using subplot to plot qt, fz, Ic, Vs with depth for the selected traveltimeMeta_ID
        fig, axs = plt.subplots(1, 2, figsize=(6, 5), dpi=100)

        axs[0].scatter(TT, TT_depth_plot, label='Travel Time Data', color='black', s=10)

        colors = plt.get_cmap('tab10').colors

        # Plot each segment with a different color
        for i in range(len(results['breaks']) - 1):
            # Define segment depth range
            z_start = results['breaks'][i]
            z_end = results['breaks'][i + 1]

            # Get depths in this segment (new_depths ensures breaks are included)
            segment_mask = (results['new_depths'] >= z_start) & (results['new_depths'] <= z_end)
            segment_depths = results['new_depths'][segment_mask]
            segment_tt = results['fitted_values_for_plot'][segment_mask]

            # Plot segment
            axs[0].plot(segment_tt, segment_depths, color=colors[i], 
                        # label=f'Segment {i+1}'
                        )

        axs[0].set_xlabel('Travel Time (ms)')
        axs[0].set_xlim(0, np.max(TT_time_plot) * 1.1)
        if np.max(TT_time_plot) > 150:
            axs[0].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 50))
        elif np.max(TT_time_plot) > 90:
            axs[0].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 30))
        elif np.max(TT_time_plot) > 50:
            axs[0].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 20))
        else:
            axs[0].set_xticks(np.arange(0, np.max(TT_time_plot) * 1.1, 10))
        axs[0].set_title('Travel Time vs Depth')
        axs[0].grid(True, which="both", ls="--")
        axs[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

        axs[1].plot(interleave(velocity, velocity), interleave(top_depth, bottom_depth), label='Direct Interpreted Velocity', color='blue')

        axs[1].set_xlim(0, 600)
        axs[1].set_xticks(np.arange(0, 601, 100))

        axs[1].set_xlabel('Velocity (m/s)')
        axs[1].set_title('Velocity vs Depth')
        axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
        axs[1].grid(True, which="both", ls="--")

        # Share common y-axis limits across all subplots
        y_max = np.nanmax(bottom_depth)
        for ax in axs:
            ax.set_ylim(y_max + 1, 0)  # inverted depth axis, same range for all

        plt.subplots_adjust(bottom=0.28, wspace=0.35)
        plt.tight_layout()
        plt.show()

        # Calculate VsZ for direct interpretation
        top_depth_dir = np.asarray(top_depth)
        top_depth_dir[0] = 0.0
        VsZ = np.max(np.asarray(bottom_depth)) / np.sum((np.asarray(bottom_depth) - top_depth_dir) / np.asarray(velocity))
        print(f"Directly Interpreted VsZ: {VsZ:.2f} m/s to depth {np.max(np.asarray(bottom_depth)):.2f} m")

        # # Calculate VsZ for direct interpretation between the first TT_depth_plot to the maximum depth of the interpreted velocity model
        top_depth_lim = np.asarray(top_depth)
        top_depth_lim [0] = np.min(TT_depth_plot)
        VsZp = (np.max(np.asarray(TT_depth_plot))-np.min(np.asarray(TT_depth_plot))) / np.sum((np.asarray(bottom_depth) - top_depth_lim) / np.asarray(velocity))
        print(f"Directly Interpreted VsZ from the top depth of the interpreted velocity model: {VsZp:.2f} m/s to depth {np.max(np.asarray(TT_depth_plot)):.2f} m")

        VsZ_input.value = VsZ
        VsZp_input.value = VsZp

        velocity_updated = velocity
        top_depth_updated = top_depth
        bottom_depth_updated = bottom_depth



# Accept Function
def Accept (_):
    
    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated

    # Save VsZ and velocity profile for the accepted model
    travelTimeMeta_ID = TRAVELTIME.value
    VsZ = VsZ_input.value
    VsZp = VsZp_input.value
    comment = COMMENT.value

    # Update the DataFrame
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['status']] = ['accepted']
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['VsZ_Direct_interpretation']] = VsZ
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['VsZp_Direct_interpretation']] = VsZp
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['comment']] = comment

    # Save to CSV
    REVIEW_DF.to_csv('Traveltime_review.csv', index=False)

    try:
        VS_SLOPEBREAK_DF = pd.read_csv('SLOPEBREAK_Vs.csv')
    except FileNotFoundError:
        VS_SLOPEBREAK_DF = pd.DataFrame(columns=['travelTimeMeta_ID', 'Velocity', 'Top_depth', 'Bottom_depth'])

    velocity_save = velocity_updated
    top_depth_save = top_depth_updated
    bottom_depth_save = bottom_depth_updated


    # Save the Vs profile to a CSV file
    TT_ID_array2 = np.full(len(velocity_save), travelTimeMeta_ID)
    new_data2 = pd.DataFrame({'travelTimeMeta_ID': TT_ID_array2,
    'Velocity': velocity_save,
    'Top_depth': top_depth_save,
    'Bottom_depth': bottom_depth_save})
    VS_SLOPEBREAK_DF = pd.concat([VS_SLOPEBREAK_DF, new_data2], ignore_index=True)
    # Save to CSV
    VS_SLOPEBREAK_DF.to_csv('SLOPEBREAK_Vs.csv', index=False)

    # Move to the next ID
    try:
        next_index = REVIEW_DF[REVIEW_DF['status'] == 'Pending']['travelTimeMeta_ID'].values[0]
        TRAVELTIME.value = next_index
    except:
        print("No more IDs to process.")

def Reject (_):
    # Update the DataFrame
    travelTimeMeta_ID = TRAVELTIME.value
    comment = COMMENT.value
    # Update the status and comment: invalid CPT or Traveltime data
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['status', 'comment']] = ['rejected', comment]
    # Save to CSV
    REVIEW_DF.to_csv('Traveltime_review.csv', index=False)


# Initial display
display(SEARCH)
display(out)

# Trigger first update manually
Update_SCPT(None)

# Attach trigger to dropdown
TRAVELTIME.observe(Update_SCPT, names=['value'])
ACCEPT.on_click(Accept)
REVISE_FULL_HAND.on_click(Full_Manual)
REJECT.on_click(Reject)

c:\Users\5_KP\anaconda3\lib\site-packages\jupyter_client\session.py:718: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


Output()